# PyRestore 03 — Adding a task means adding a definition, not editing code

This notebook replays the real Shenzhen six-dimension task through the same `run_task` API used by the PRS case. It uses the frozen, authorized model responses and precomputed CV table, so re-execution is offline and does not require an API key or transmit imagery.

The evidence comprises 465 coordinate-matched images with human ratings for Safe, Lively, Beautiful, Wealthy, Boring and Depressing.

In [1]:
from pathlib import Path

from pyrestore import load_config, load_task, run_task

root = Path.cwd()
prs11 = load_task(root / "tasks/prs11.yaml")
sixdim = load_task(root / "tasks/sixdim.yaml")
print("PRS-11 fields:", list(prs11["output_fields"]))
print("Six-dimension fields:", list(sixdim["output_fields"]))
print("Different prompt templates:", prs11["prompt"]["template"] != sixdim["prompt"]["template"])

PRS-11 fields: ['being_away', 'coherence', 'scope', 'fascination']
Six-dimension fields: ['safe', 'lively', 'beautiful', 'wealthy', 'boring', 'depressing']
Different prompt templates: True


## 1. Offline replay of the real six-dimension run

The precomputed tables retain the source model, prompt hash, request hash and row-level status. PyRestore checks those fields against the current task contract before publishing a new managed output directory.

In [2]:
data = root / "data/shenzhen_sixdim"
manifest = data / "manifest.csv"
cv_table = data / "cv_features.csv"
vlm_table = data / "vlm_scores.csv"
reference = data / "reference.csv"

cfg = load_config(root / "config/submission.yaml")
result = run_task(
    manifest,
    root / "tasks/sixdim.yaml",
    cfg,
    cv_features=cv_table,
    vlm_scores=vlm_table,
    reference=reference,
    check_files=False,   # raw Shenzhen imagery is not redistributed; precomputed tables only
    make_map=False,
    out_dir=root / "outputs/notebook/urban_perception_sixdim",
)
result["quality"]

,n_input_rows,n_valid_coordinates,n_cv_success,n_cv_error,n_vlm_success,n_vlm_error
0,465,465,465,0,465,0


In [3]:
columns = ["target", "n", "pearson_r", "pearson_ci_low", "pearson_ci_high", "spearman_rho", "status"]
result["validation"][columns].round(3)

,target,n,pearson_r,pearson_ci_low,pearson_ci_high,spearman_rho,status
0,vlm_safe,465,0.259,0.172,0.342,0.216,validated
1,vlm_lively,465,-0.054,-0.145,0.037,-0.041,validated
2,vlm_beautiful,465,0.573,0.509,0.631,0.552,validated
3,vlm_wealthy,465,0.407,0.328,0.480,0.377,validated
4,vlm_boring,465,0.212,0.123,0.297,0.213,validated
5,vlm_depressing,465,0.470,0.396,0.538,0.489,validated


## 2. Interpretation

The replay establishes that adding the six-dimension task required a YAML definition and prompt resource rather than framework changes. All 465 records completed the declared schema, and the same product family contains quality, validation, provenance and GIS outputs.

Validity remains dimension-specific. Beautiful has the strongest Pearson agreement with human ratings (about 0.51), whereas Lively is weak (about 0.14). The result supports task configurability and a real validation workflow; it does not support a claim that the model reproduces every human-perception dimension equally well.